# Reset-Safe Notebook: Foundry + OTEL + LangSmith

Every cell can be re-run without restarting the kernel. The TracerProvider is created once and reused.

## 1. Install Dependencies

In [ ]:
%pip install azure-ai-projects azure-ai-agents azure-identity azure-core-tracing-opentelemetry opentelemetry-sdk opentelemetry-exporter-otlp-proto-http python-dotenv --quiet

## 2. Define Environment Variables

You will be prompted to enter your personal values. These stay in-memory only.

In [ ]:
import os
from getpass import getpass

print("Enter your Azure AI Foundry and LangSmith credentials below.")
print("(Find PROJECT_ENDPOINT in Azure AI Foundry portal -> Overview page)")
print("(Find LANGSMITH_API_KEY at smith.langchain.com -> Settings -> API Keys)")
print()

os.environ["PROJECT_ENDPOINT"] = input("PROJECT_ENDPOINT (e.g. https://myproject.services.ai.azure.com): ").strip()
os.environ["MODEL_DEPLOYMENT_NAME"] = input("MODEL_DEPLOYMENT_NAME (e.g. gpt-4o): ").strip() or "gpt-4o"
os.environ["LANGSMITH_API_KEY"] = getpass("LANGSMITH_API_KEY (hidden): ").strip()

# These have sensible defaults
os.environ["LANGSMITH_OTEL_ENDPOINT"] = "https://api.smith.langchain.com/otel/v1/traces"
os.environ["OTEL_SERVICE_NAME"] = "foundry-otel-langsmith"
os.environ["DEPLOYMENT_ENVIRONMENT"] = "development"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print()
print(f"OK - PROJECT_ENDPOINT: {os.environ['PROJECT_ENDPOINT']}")
print(f"OK - MODEL_DEPLOYMENT_NAME: {os.environ['MODEL_DEPLOYMENT_NAME']}")
print(f"OK - LANGSMITH_API_KEY: {'*' * len(os.environ['LANGSMITH_API_KEY'])}")
print(f"OK - LANGSMITH_OTEL_ENDPOINT: {os.environ['LANGSMITH_OTEL_ENDPOINT']}")

## 3. Initialize Telemetry (reset-safe)

This cell is safe to re-run. It only creates the TracerProvider once.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from azure.core.settings import settings
from azure.ai.agents.telemetry import AIAgentsInstrumentor

# Only initialize once - safe to re-run this cell
if not isinstance(trace.get_tracer_provider(), TracerProvider):
    settings.tracing_implementation = "opentelemetry"

    resource = Resource.create({
        "service.name": os.environ["OTEL_SERVICE_NAME"],
        "deployment.environment": os.environ["DEPLOYMENT_ENVIRONMENT"],
    })

    provider = TracerProvider(resource=resource)
    exporter = OTLPSpanExporter(
        endpoint=os.environ["LANGSMITH_OTEL_ENDPOINT"],
        headers={"x-api-key": os.environ["LANGSMITH_API_KEY"]},
    )
    provider.add_span_processor(BatchSpanProcessor(exporter))
    trace.set_tracer_provider(provider)

    AIAgentsInstrumentor().instrument()
    print("OK - TracerProvider created and AIAgentsInstrumentor activated")
else:
    provider = trace.get_tracer_provider()
    print("OK - TracerProvider already initialized (reusing existing)")

tracer = trace.get_tracer("notebook")

## 4. Test Azure Authentication

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
token = credential.get_token("https://management.azure.com/.default")
print(f"OK - Authenticated (token expires: {token.expires_on})")

## 5. Test AgentsClient Connection

In [ ]:
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

agents_client = AgentsClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

print(f"OK - AgentsClient connected to: {os.environ['PROJECT_ENDPOINT']}")
print(f"   Available: create_agent, create_thread_and_process_run, delete_agent")
agents_client.close()

## 6. Test Span Export to LangSmith

Sends a small test span. Check your LangSmith dashboard after running.

In [ ]:
with tracer.start_as_current_span("notebook.test_span") as span:
    span.set_attribute("test.source", "reset-safe-notebook")
    span.set_attribute("test.step", "verify_export")

provider.force_flush()
print("OK - Test span sent to LangSmith")
print("   -> Look for 'notebook.test_span' in your dashboard")

## 7. Run Agent End-to-End

Change the `PROMPT` variable and re-run this cell as many times as you want.

In [ ]:
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import CodeInterpreterTool, ListSortOrder, MessageRole, AgentThreadCreationOptions
from azure.identity import DefaultAzureCredential

PROMPT = "Write a python function to find odd numbers in any input. Show example of it working."

with tracer.start_as_current_span("notebook.agent_run") as root_span:
    root_span.set_attribute("prompt", PROMPT)

    agents_client = AgentsClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    code_interpreter = CodeInterpreterTool()

    with agents_client:
        agent = agents_client.create_agent(
            model=os.environ["MODEL_DEPLOYMENT_NAME"],
            name="notebook-code-interpreter",
            instructions="You are a helpful Python coding assistant. Write and execute code to answer questions.",
            tools=code_interpreter.definitions,
        )
        print(f"OK - Agent created: {agent.id}")

        try:
            thread_options = AgentThreadCreationOptions(
                messages=[{"role": "user", "content": PROMPT}]
            )
            print("Running agent...")

            run = agents_client.create_thread_and_process_run(
                agent_id=agent.id,
                thread=thread_options,
            )
            root_span.set_attribute("run.status", run.status)
            print(f"OK - Run status: {run.status}")

            if run.status == "failed":
                print(f"FAILED: {run.last_error}")
            else:
                messages = agents_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.DESCENDING)
                for msg in messages:
                    if msg.role == MessageRole.AGENT and msg.text_messages:
                        print(f"\n{'='*60}")
                        print("AGENT RESPONSE:")
                        print(f"{'='*60}")
                        print(msg.text_messages[-1].text.value)
                        break
        finally:
            agents_client.delete_agent(agent.id)
            print("\nAgent cleaned up")

provider.force_flush()
print("\nTraces flushed to LangSmith")

## 8. (Optional) Shutdown

Only run this when completely done. After shutdown you need to restart the kernel.

In [ ]:
# provider.shutdown()
# print("TracerProvider shut down. Restart kernel to use again.")